# 09_01 Passages, and a word search from scratch (BM25)

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: cutting the book into pages, and teaching the librarian to fetch
>
> Project files this connects to:
>
> - `src/homer_ai/retrieval.py`
> - `data/homer/`
>
> **This notebook:** Before anything can be looked up, the poems must be cut into passages that know where they came from, and we need a score that says which passage fits a question.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to cut a text into passages that remember their book and line
- How to turn text into a bag of words, and why counting words isn't enough
- What tf, idf and BM25 are, worked out by hand on 3 real passages
- How an index makes the search fast on all 2,140 passages

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **chunking** | cutting a long text into passages | Book 1 of the Odyssey: 444 verses → 89 passages of 5 |
| **bag of words** | a text seen only as its words and how often each appears; the order is thrown away | `noman: 3, comrades: 1, …` |
| **tf** | term frequency: how often a word is in one passage | `noman` is 3 times in `odyssey 9:25` |
| **idf** | inverse document frequency: how much a word is worth; high when few passages have it | `pigs` is worth more than `the` |
| **BM25** | a formula that scores a passage against the words of a question, from tf, idf and the passage's length | what search engines used long before neural networks |
| **inverted index** | a table that lists, for each word, the passages that contain it | `moly → [odyssey 10:24]` |

## 3. The idea

### ❓ What does the librarian need before the first question?

Two things. Pages to fetch, and a way to decide which page fits a question.

The pages are **passages**: short pieces of the poems, each one tagged with where it came from. Without the tag there's no citation later.

The way to decide is a score. For a question and a passage, we want one number that is high when the passage fits. We build that number in three tries, each fixing what the one before got wrong:

```
1. count the question's words in the passage      (bag of words)
2. make rare words worth more                     (idf)
3. stop repeats and long passages from winning    (BM25)
```

In the picture of [09_00](09_00_chapter_overview.ipynb), this notebook cuts the book into pages and teaches the librarian to fetch.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what do the files look like?

Each book is one text file. In the Greek files a line is one verse. In Butler's English, which is prose, a line is one whole paragraph. [07_01](../07_advanced_iliad_and_combining/07_01_the_corpus.ipynb) looked at these files; here we only need the line counts.

👉 **Predict first:** Book 9 of the Odyssey (the Cyclops) has 566 Greek verses. Roughly how many English lines?

In [ ]:
from homer_ai.retrieval import HOMER

greek = (HOMER / "odyssey" / "greek" / "book_09.txt").read_text(encoding="utf-8")
english = (HOMER / "odyssey" / "english" / "book_09.txt").read_text(encoding="utf-8")
greek_lines, english_lines = greek.splitlines(), english.splitlines()

print(f"{'':8} {'lines':>6} {'characters in an average line':>30}")
for name, lines in (("greek", greek_lines), ("english", english_lines)):
    print(f"{name:8} {len(lines):>6} {sum(map(len, lines)) / len(lines):>30.0f}")
print()
print("greek   line 1:", greek_lines[0])
print("english line 1:", english_lines[0][:60] + "…")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

566 Greek lines of about 44 characters, and only 44 English lines of about 670. An English line is a paragraph, well over 100 words. A Greek line is one verse, around 7 words.

So the two languages need different cuts. A paragraph is already a good passage. One verse is too little to search or to answer from.

</details>

### ❓ Step 2: how do I cut lines into passages?

Take the lines in runs of a fixed size, and write down the first and last line number of each run. For the Greek we use 5 verses.

Here are the first 12 verses of the Odyssey, cut by 5:

```
lines   :  1  2  3  4  5 |  6  7  8  9 10 | 11 12
passage :  odyssey 1:1-5 | odyssey 1:6-10 | odyssey 1:11-12
```

The last run is short, and that's fine. The reference `odyssey 1:6-10` reads: the Odyssey, Book 1, lines 6 to 10 of our file.

👉 **Predict first:** all of Book 1 has 444 verses. How many passages of 5 is that?

In [ ]:
def chunk_lines(lines, size):
    chunks = []
    for start in range(0, len(lines), size):
        part = lines[start : start + size]
        chunks.append((start + 1, start + len(part), "\n".join(part)))
    return chunks


book1 = (HOMER / "odyssey" / "greek" / "book_01.txt").read_text(encoding="utf-8")
book1_lines = book1.splitlines()
for first, last, text in chunk_lines(book1_lines[:12], 5):
    print(f"{f'odyssey 1:{first}-{last}':16} {text.splitlines()[0]}")
print()
print("all of Book 1:", len(chunk_lines(book1_lines, 5)), "passages")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Three passages from 12 lines, as in the picture, each shown by its first verse. All of Book 1 gives 89: that's 88 full ones (440 verses) and one of 4.

`start + 1` is there because Python counts from 0 and people count lines from 1.

</details>

### ❓ Step 3: what about the end of a book?

A passage never crosses it. We cut each book's file by itself, so the last passage of Book 1 is short, and the first passage of Book 2 starts at line 1.

If a passage ran from the end of one book into the next, its reference would be wrong, and it would mix two scenes that don't belong together. It's the same reason [02_02](../02_text_to_numbers/02_02_special_tokens.ipynb) put `<|endoftext|>` between books.

`load_passages` in `retrieval.py` does this for a whole poem: one line per passage for the English, 5 for the Greek. Each passage is a small record with `poem`, `language`, `book`, `first`, `last`, `text`, and a `ref`.

In [ ]:
from homer_ai.retrieval import load_passages

print(f"{'':10} {'english':>8} {'greek':>8}")
for poem in ("odyssey", "iliad"):
    counts = [len(load_passages(poem, language)) for language in ("english", "greek")]
    print(f"{poem:10} {counts[0]:>8,} {counts[1]:>8,}")

passages = load_passages("odyssey", "english")
passage = passages[1]  # the second paragraph of Book 1
print()
print(passage.ref, "|", passage.text[:70] + "…")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

1,027 English passages for the Odyssey and 1,113 for the Iliad: 2,140 together, one per paragraph. The Greek gives 2,432 and 3,149.

The record for `odyssey 1:2` is the paragraph that begins *So now all who escaped death in battle…*. That reference is what Claude will cite later.

</details>

### ❓ Step 4: how does text become words?

Lowercase it, drop the accents, and keep runs of letters. That's `tokenize` in `retrieval.py`. (It's a much simpler job than the tokenizers of chapter 2. Here a token is a whole word, and nothing has to be turned back into text.)

Dropping accents matters for the Greek. `μῆνιν` and `μηνιν` should be the same word to a search, and nobody wants to type breathings into a question.

👉 **Predict first:** what happens to the `’s` in `Noman’s`?

In [ ]:
from homer_ai.retrieval import tokenize

print(tokenize("‘Then I will eat all Noman’s comrades before Noman himself’"))
print(tokenize("μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος"))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The quote marks are gone, everything is lowercase, and `Noman’s` became two words, `noman` and `s`. That lone `s` is noise, but harmless: it's in so many passages that it will count for almost nothing. You'll see why in step 6.

The Greek lost its accents and breathings: `μηνιν`, `αειδε`, `θεα`.

</details>

### ❓ Step 5: can't we just count the question's words?

Let's try, on three real passages. Two are from the Cyclops story, where Odysseus has said his name is *Noman*. The third is from Circe's island.

```
odyssey 9:25   ‘Then I will eat all Noman’s comrades before Noman himself, and will keep Noman for the last. …’
odyssey 9:28   ‘Noman is killing me by fraud; no man is killing me by force.’
odyssey 10:18  … Circe threw them some acorns and beech masts such as pigs eat, but Eurylochus hurried back to tell me about the sad fate of our comrades. …
```

The question is *who is killing Noman?* We look at two of its words, `noman` and `killing`, and count them in each passage. A text seen this way, as words and counts with the order thrown away, is called a **bag of words**. The count of one word in one passage is its **tf**, term frequency.

👉 **Predict first:** the passage that answers is `odyssey 9:28`. Will it get the highest count?

In [ ]:
from homer_ai.retrieval import load_passages, tokenize

by_ref = {p.ref: p for p in load_passages("odyssey", "english")}
toy = [by_ref["odyssey 9:25"], by_ref["odyssey 9:28"], by_ref["odyssey 10:18"]]
toy_words = [tokenize(p.text) for p in toy]

query = ["noman", "killing"]
print(f"{'':14} {'noman':>6} {'killing':>8} {'sum':>5} {'words':>6}")
for p, words in zip(toy, toy_words, strict=True):
    tf = [words.count(word) for word in query]
    print(f"{p.ref:14} {tf[0]:>6} {tf[1]:>8} {sum(tf):>5} {len(words):>6}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A tie. `odyssey 9:25` says `noman` three times and gets 3. `odyssey 9:28`, the one that answers, has `noman` once and `killing` twice, and also gets 3.

Plain counting treats every word the same. But `killing` is the word that tells these passages apart, and `noman` is in both. The count should know that.

</details>

### ❓ Step 6: how do I make a rare word worth more?

Count in how many passages each word appears, and turn that into a weight. A word that's in few passages gets a high weight. A word that's in nearly all of them gets a weight near 0. This weight is the **idf**, inverse document frequency (a *document* is a passage here).

With `N` passages in all, and `n` of them containing the word:

```
idf = log(1 + (N - n + 0.5) / (n + 0.5))
```

Read it from the inside. `(N - n) / n` is *passages without the word* divided by *passages with it*: big for a rare word. The two `0.5`s keep the division safe when `n` is 0. `log` (the natural logarithm, met in [05_01](../05_training_on_the_odyssey/05_01_loss_and_epochs.ipynb)) squeezes the result, so a word that's 100 times rarer isn't worth 100 times more. And the `1 +` keeps the weight above 0.

With our 3 passages:

| word | in how many of 3 | the maths | idf |
|---|---|---|---|
| `noman` | 2 | log(1 + 1.5 / 2.5) | 0.47 |
| `killing` | 1 | log(1 + 2.5 / 1.5) | 0.98 |
| `the` | 3 | log(1 + 0.5 / 3.5) | 0.13 |

Then the score of a passage is, for each question word, `tf × idf`, added up. That's **TF-IDF**.

In [ ]:
import math

N = len(toy)


def idf(word):
    n = sum(word in words for words in toy_words)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))


for word in ("noman", "killing", "the"):
    print(f"idf({word}) = {idf(word):.2f}")
print()
for p, words in zip(toy, toy_words, strict=True):
    parts = [words.count(word) * idf(word) for word in query]
    print(f"{p.ref:14} {parts[0]:.2f} + {parts[1]:.2f} = {sum(parts):.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same three idf values as in the table. And the tie is broken: `odyssey 9:28` scores 0.47 + 1.96 = 2.43, ahead of `odyssey 9:25` with 1.41. The passage with the telling word wins.

Look at `the`: 0.13. Words that are everywhere fade out by themselves, so we never need a list of words to ignore. (That's also what happens to the stray `s` of step 4.)

</details>

### ❓ Your turn: can you score a passage for a new question?

A new question, two words: `noman` and `comrades`. Work out the TF-IDF score of the first toy passage (`odyssey 9:25`) for it.

👉 **Predict first:** `noman` is in this passage 3 times and `comrades` once. Both have an idf of 0.47. What's the score?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
new_query = ["noman", "comrades"]

# 👉 do: the TF-IDF score of toy_words[0] for new_query: tf × idf per word, added up
# 💡 hint: toy_words[0] is the passage as a list of words, and a list can count() a word
tfidf_score = None

In [ ]:
from homer_ai.lessons import check

check(
    "TF-IDF score of odyssey 9:25",
    tfidf_score,
    3 * idf("noman") + 1 * idf("comrades"),
    coach="Got 0.94? That's the two idf values without the counts: noman is in "
    "this passage 3 times, so its idf counts 3 times. Got 4? Those are the "
    "counts without the idf.",
)

<details><summary>💡 Hint</summary>

The last loop of the cell above does it for every passage: `words.count(word) * idf(word)` for each word of the query. Here `words` is `toy_words[0]`.

</details>

<details><summary>✅ Answer</summary>

```python
new_query = ["noman", "comrades"]

tfidf_score = sum(toy_words[0].count(word) * idf(word) for word in new_query)
```

`3 × 0.47 + 1 × 0.47 = 1.88`. Notice that three `noman`s count exactly three times as much as one. Is the third mention really worth as much as the first? That's the first thing BM25 changes.

</details>

### ❓ Step 7: what's still wrong, and what does BM25 change?

Two things are still wrong with `tf × idf`.

**Repeats count too much.** A passage that says `noman` three times gets three times the points of one that says it once. But it isn't three times more about Noman. The first mention tells you a lot, and the third tells you little.

**Long passages win too easily.** A paragraph of 400 words has more chances to contain any word than one of 20 words. That makes it longer. It doesn't make it a better fit.

BM25 keeps the idf and swaps the plain `tf` for this:

```
            tf × (k1 + 1)
idf  ×  ─────────────────────────────────────────────
         tf + k1 × (1 - b + b × length / average)
```

| Piece | What it does |
|---|---|
| `tf` above and below the line | the fraction grows with `tf`, but ever more slowly; it can never pass `k1 + 1` |
| `k1` (we use 1.5) | how fast repeats stop counting; a smaller `k1` flattens sooner |
| `length / average` | the passage's word count against the average passage; 1 for an average one |
| `b` (we use 0.75) | how much length matters; with `b = 0` length is ignored |

For a passage of average length the bracket is exactly 1, and the fraction is `tf × 2.5 / (tf + 1.5)`: 1.0 for one mention, 1.43 for two, 1.67 for three. Let's draw it.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    BAD,
    GOOD,
    INK,
    MUTED,
    studio_axes,
    studio_figure,
)

K1, B = 1.5, 0.75


def part(tf, length, average=100):
    return tf * (K1 + 1) / (tf + K1 * (1 - B + B * length / average))


counts = range(0, 11)
fig = studio_figure("BM25", "The fifth Noman counts for less than the first")
ax = studio_axes(fig)
ax.plot(counts, list(counts), color=MUTED, linewidth=2, linestyle="--")
ax.plot(counts, [part(tf, 50) for tf in counts], "o-", color=GOOD, linewidth=2.5)
ax.plot(counts, [part(tf, 100) for tf in counts], "o-", color=ACCENT, linewidth=2.5)
ax.plot(counts, [part(tf, 300) for tf in counts], "o-", color=BAD, linewidth=2.5)
ax.axhline(K1 + 1, color=MUTED, linewidth=1, linestyle=":")
ax.text(10, 2.62, "the ceiling: k1 + 1 = 2.5", color=MUTED, fontsize=9, ha="right")
ax.text(3.2, 4.0, "plain counting", color=MUTED, fontsize=10)
labels = [
    (part(10, 50) + 0.06, "short passage", GOOD),
    (part(10, 100) - 0.08, "average length", ACCENT),
    (part(10, 300), "3 times longer", BAD),
]
for y, text, color in labels:
    ax.text(10.2, y, text, color=color, fontsize=9.5, va="center")
ax.set_xlim(-0.3, 12.6)
ax.set_ylim(0, 4.5)
ax.set_xlabel("how often the word is in the passage (tf)")
ax.set_ylabel("what it adds to the score (before idf)")
ax.tick_params(colors=INK)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The dashed line is plain counting: every repeat adds the same. The three curves are BM25. They rise fast at first and then flatten under the dotted ceiling at 2.5.

The green curve is a passage half the average length, the orange one is average, and the pink one is 3 times longer. For the same count, the long passage always earns less.

</details>

### ❓ Step 8: does it give those numbers on our three passages?

We write the formula as a function, score the three passages, and then let the finished `BM25` class from `retrieval.py` do the same. The two must agree.

This time we score three question words: `noman`, `killing` and `comrades`. The word `comrades` is once in `odyssey 9:25` (32 words) and once in `odyssey 10:18` (86 words).

👉 **Predict first:** same word, same count. Which of those two passages gets more points for `comrades`?

In [ ]:
from homer_ai.retrieval import BM25

K1, B = 1.5, 0.75
average = sum(len(words) for words in toy_words) / N


def bm25_part(word, words):
    tf = words.count(word)
    length = 1 - B + B * len(words) / average
    return idf(word) * tf * (K1 + 1) / (tf + K1 * length)


query = ["noman", "killing", "comrades"]
index = BM25([p.text for p in toy])  # the finished class
from_class = index.scores(query)

print(f"{'':14} {'noman':>6} {'killing':>8} {'comrades':>9} {'sum':>6} {'class':>6}")
for number, (p, words) in enumerate(zip(toy, toy_words, strict=True)):
    parts = [bm25_part(word, words) for word in query]
    print(
        f"{p.ref:14} {parts[0]:>6.2f} {parts[1]:>8.2f} {parts[2]:>9.2f} "
        f"{sum(parts):>6.2f} {from_class.get(number, 0):>6.2f}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The last two columns, `sum` and `class`, match, so the class does what our hand-written function does.

Read the `noman` column first. Three mentions in `odyssey 9:25` earn 0.85, and one mention in `odyssey 9:28` earns 0.62. Three times the count gives about a third more points. That's the flattening (the second passage is also a little shorter, which helps it).

Now `comrades`: 0.55 in the short passage, 0.34 in the long one, for the same single mention. That's the length part.

`odyssey 9:28` still wins overall with 2.30, thanks to `killing`.

</details>

### ❓ Step 9: how does this stay fast on 2,140 passages?

By not looking at most of them. When the index is built, we go through every passage once and fill a table: for each word, the passages that contain it and how often. It's called an **inverted index**, and it's the same idea as the index at the back of a book.

```
moly    → [(odyssey 10:24, 1 time)]
sirens  → [(odyssey 12:5, 2 times), (odyssey 12:14, 1 time), …]
the     → nearly every passage
```

For a question we only visit the rows of its own words, and add up the BM25 parts per passage. A rare word touches a handful of passages. That table is `BM25.postings`.

In [ ]:
import time

both = load_passages("odyssey", "english") + load_passages("iliad", "english")
started = time.perf_counter()
index = BM25([p.text for p in both])
print(f"{len(both):,} passages indexed in {time.perf_counter() - started:.2f} seconds")
print(f"{len(index.postings):,} different words")
print(f"an average passage has {index.average:.0f} words")
print()
for word in ("moly", "sirens", "noman", "ulysses", "the"):
    n = len(index.postings[word])
    print(f"{word:8} in {n:>5,} passages   idf {index.idf(word):.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About a quarter of a second to index both poems, and that happens once. There are 9,646 different words, and an average passage has 127.

`moly` is in a single passage, so its idf is high (7.26). `the` is in 2,049 of 2,140 and is worth 0.04. Look at `ulysses`: 477 passages, idf 1.50. The hero's name is so common in this text that it hardly helps a search. Remember that for the next notebook.

</details>

### ❓ Step 10: does it find the right passage?

Let's ask a real question: *What false name does Odysseus give to the Cyclops?* `index.search` returns the best passage numbers with their scores.

👉 **Predict first:** which words of the question will do the work?

In [ ]:
question = "What false name does Odysseus give to the Cyclops?"
question_words = tokenize(question)
print(f"{'word':10} {'idf':>5}")
for word in question_words:
    print(f"{word:10} {index.idf(word):>5.2f}")
print()
for number, score in index.search(question_words, k=3):
    p = both[number]
    print(f"{score:5.1f}  {p.ref:14} {p.text[:66]}…")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first hit is `odyssey 9:24`, the paragraph where he says *my name is Noman*. That's the right one.

The idf column shows why. `false`, `cyclops` and `name` carry the search, and `to` and `the` are worth next to nothing. (`does` gets 3.69, more than `name`. Butler rarely writes *does*. idf knows how rare a word is. It doesn't know whether the word means anything.)

And `odysseus` has the highest idf of all, 8.36. That's a warning, not good news: it's what a word gets when **no passage contains it**. Butler's English never says Odysseus. [09_02](09_02_search_and_scores.ipynb) starts there.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why 5 verses per Greek passage? Why not 3, or 20?

It's a trade. A small passage gives a precise citation but may not hold enough to answer from, and its few words make a thin target for a search. A big one is easier to find and blurrier to cite. Five verses is about one sentence or two of Homer. With labelled questions you can try other sizes and measure, which is what 09_02 is about.

### ❓ Real systems let passages overlap. Why don't we?

Overlap helps when an answer sits across a cut: with a shared verse or two, one of the passages holds all of it. It also makes the index bigger and returns near-duplicates. We leave it out to keep the line numbers simple. `chunk_lines` would need one more argument, the step between starts.

### ❓ Shouldn't `kills`, `killed` and `killing` count as the same word?

Cutting words down to a stem is called stemming, and many search engines do it. I tried a simple one here (strip `-s`, `-ed`, `-ing`) and the scores on our questions didn't improve, so it's not in the code. That's the useful habit: measure before you add a part.

### ❓ Where do `k1 = 1.5` and `b = 0.75` come from?

They're the usual starting values, found by people who tested BM25 on many collections. On our questions, other values in the normal range (`k1` from 0.9 to 2, `b` from 0.4 to 0.9) moved the scores by a question or two at most.

## 5. Recap

- A passage is a short run of lines that remembers its poem, book and lines. It never crosses a book boundary.
- A bag of words keeps the words and their counts, and forgets the order.
- idf makes a rare word worth more; words that are everywhere fade out by themselves.
- BM25 adds two corrections: repeats count less and less, and a long passage earns less for the same count.
- An inverted index lists the passages for each word, so a question only touches the passages that share a word with it.

**You can now:**

- cut a text into passages that know their book and lines
- compute idf and a TF-IDF score by hand
- say what BM25's two corrections do, and what `k1` and `b` set

**Next:** [09_02: Searching both poems, and scoring the search](09_02_search_and_scores.ipynb).